In [22]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import detectors as det
import filters as filt
import integrators as integ

csv_file_name = 'initial_walk_test_10-08-2026_16-31-45_3.csv'
csv_path = '../data/measured_walks/'
csv_save_path = '../data/orientation_mahony/'
df = pd.read_csv(f'{csv_path}{csv_file_name}')

In [23]:
# =============
# Set parameters and detect ZVWs
# =============
acc_dev = det.ACC_DEVIATION
gyro_limit = det.GYRO_LIMIT
var_limit = det.VAR_LIMIT
var_window = det.VAR_WINDOW
dwell = det.DWELL

ax = df['ax']
ay = df['ay']
az = df['az']
gx = df['gx']
gy = df['gy']
gz = df['gz']

dt_array = np.diff(df['t_us'] - df['t_us'].iloc[0]) / 1e6
# Add a mean value at the beginning
dt_array = np.insert(dt_array, 0, dt_array.mean())

zvw_mask = det.detect_zvw(df, acc_dev, gyro_limit, var_limit, var_window, dwell)
masked_quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, zvw_mask)
masked_roll, masked_pitch, masked_yaw = filt.quaternions_to_euler(masked_quats)

no_zvw_mask = np.zeros_like(zvw_mask, dtype=bool)
unmasked_quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, no_zvw_mask)
unmasked_roll, unmasked_pitch, unmasked_yaw = filt.quaternions_to_euler(unmasked_quats)


In [24]:
# =============
# Plot Roll and pitch
# =============

time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.plot(time_sec, unmasked_roll, label='Gyro Only (Drifting)', color='tab:red', alpha=0.6, linewidth=1)
ax1.plot(time_sec, masked_roll, label='ZVW Gated Mahony (Stable)', color='tab:blue', linewidth=1.5)
ax1.set_title('Roll (X-Axis Tilt): Gyro Drift vs. ZVW Gated Mahony')
ax1.set_ylabel('Angle (deg)')
ax1.set_xlabel('Time (s)')
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(time_sec, unmasked_pitch, label='Gyro only (drifting)', color='tab:red', alpha=0.6, linewidth=1)
ax2.plot(time_sec, masked_pitch, label='ZVW Gated Mahony (Stable)', color='tab:blue', linewidth=1.5)
ax2.set_title('Pitch (Y-Axis Tilt): Gyro Drift vs. ZVW Gated Mahony')
ax2.set_ylabel('Angle (deg)')
ax2.set_xlabel('Time (s)')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_roll_pitch.png'), dpi=120)
plt.show()


In [25]:
# =============
# Plot ZVW Residuals
# =============

raw_accel = np.column_stack((ax, ay, az))

# Rotate the raw acceleration into the global frame
global_accel = filt.rotate_vector_by_quaternion(raw_accel, masked_quats)

linear_accel = np.copy(global_accel)
linear_accel[:, 2] -= 1.0

zvs_residuals = linear_accel[zvw_mask]

mean_x_error = np.mean(zvs_residuals[:, 0])
mean_y_error = np.mean(zvs_residuals[:, 1])
mean_z_error = np.mean(zvs_residuals[:, 2])

print(f"Mean X Error: {mean_x_error} g")
print(f"Mean Y Error: {mean_y_error} g")
print(f"Mean Z Error: {mean_z_error} g")

# Histogram Z Error
fig, ax_hist = plt.subplots(1, 1, figsize=(8, 4))
ax_hist.hist(zvs_residuals[:, 0], bins=np.arange(-0.5, 0.5, 0.01), label='X Error')
ax_hist.hist(zvs_residuals[:, 1], bins=np.arange(-0.5, 0.5, 0.01), label='Y Error')
ax_hist.hist(zvs_residuals[:, 2], bins=np.arange(-0.5, 0.5, 0.01), label='Z Error')
ax_hist.set_title('Histogram of ZVW Residuals')
ax_hist.set_xlabel('ZVW Residual (g)')
ax_hist.set_ylabel('Count')
ax_hist.grid(True, linestyle='--', alpha=0.5)
ax_hist.legend()
plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_ZVW_residuals.png'), dpi=120)
# plt.show()


Mean X Error: 0.012098666146324207 g
Mean Y Error: -0.012148534416344718 g
Mean Z Error: 0.02331376303214137 g


In [26]:
# ===========
# ZUPT
# ===========

# convert g to ms^2
accel_ms2 = linear_accel * 9.80665

# run through integrator
vel_zupt, pos_zupt, pre_zupt_vels = integ.integrate_kinematics(accel_ms2, dt_array, zvw_mask)

final_xy_dist = np.linalg.norm(pos_zupt[-1, :2])
final_z_dist = pos_zupt[-1, 2]
mean_pre_zupt_x = np.mean(pre_zupt_vels[:, 0])

print(f"Final XY Distance: {final_xy_dist:.3f} m")
print(f"Target: 19.985 m \nError percentage: {100 - (final_xy_dist * 100 / 19.985):.3f}%")
print(f"Final Z (Vertical) Error: {final_z_dist:.3f} m")
print(f"Mean Pre-ZUPT X-Velocity: {mean_pre_zupt_x:.3f} m/s")

Final XY Distance: 18.855 m
Target: 19.985 m 
Error percentage: 5.654%
Final Z (Vertical) Error: 0.357 m
Mean Pre-ZUPT X-Velocity: 0.278 m/s
